In [ ]:
import geopandas as gpd
import pandas as pd

In [ ]:
from shapely import from_wkt

def barcelona_census_areas():
    url = "BarcelonaCiutat_SeccionsCensals.csv"
    df = pd.read_csv(url)
    
    df["geometry"] = df["geometria_wgs84"].apply(from_wkt)
    df = df.drop(["geometria_etrs89","geometria_wgs84"],axis=1)
    return gpd.GeoDataFrame(df,geometry="geometry",crs="EPSG:4326")


In [ ]:
census_areas_gdf = barcelona_census_areas()

In [ ]:
census_areas_gdf

In [ ]:
census_areas_gdf.explore("codi_seccio_censal")

In [ ]:
census_areas_gdf.explore("nom_barri")

Some ideas for things that come under "socio-economic structure" (per region):
* population size
* age distribution
* average wage / income
* employment rate
* health
* economic activity (number of shops, type of shops)
* migration in/out patterns
* home ownership rates and types of homes
* education
* access to services / facilities

In [ ]:
# Number of deaths in Barcelona by year by the the Municipal Register of Inhabitants
# https://opendata-ajuntament.barcelona.cat/data/en/dataset/pad_def_mdbas/resource/8f03667c-863b-4507-a6cd-070b868dd941
deaths_df = pd.read_csv("2024_pad_def_mdbas.csv")
deaths_df

In [ ]:
# the file appears to maybe have "Seccio_Censal" encoded as "nom_districte" prepended on "codi_seccio_censal"?

In [ ]:
deaths_df[deaths_df["Seccio_Censal"].isin([10143,1005])]

In [ ]:
census_areas_gdf[census_areas_gdf["codi_seccio_censal"].isin([143,5])]

In [ ]:
# ah, so it looks like codi_seccio_censal in census_areas_gdf isn't unique, and 
# full code for a census area needs to have the district prepended

In [ ]:
census_areas_gdf["Seccio_Censal"] = (
    (census_areas_gdf["codi_districte"] * 1000 ) + census_areas_gdf["codi_seccio_censal"]
)

In [ ]:
census_areas_gdf.explore("Seccio_Censal")

In [ ]:
census_areas_gdf["codi_seccio_censal"].is_unique

In [ ]:
census_areas_gdf["Seccio_Censal"].is_unique

In [ ]:
deaths_gdf = census_areas_gdf.merge(deaths_df, on="Seccio_Censal")
deaths_gdf

In [ ]:
deaths_gdf.explore("Valor")

In [ ]:
deaths_gdf["deaths"] = deaths_gdf["Valor"].astype(float)

In [ ]:
deaths_gdf[deaths_gdf["Valor"] == ".."]

In [ ]:
deaths_gdf[deaths_gdf["Valor"] == ".."].count()

In [ ]:
deaths_gdf[deaths_gdf["Valor"] != ".."].count()

In [ ]:
deaths_gdf["deaths"] = (
    deaths_gdf["Valor"]
        .str.replace("..","0")
        .astype(float)
)

In [ ]:
deaths_gdf.explore("deaths")

In [ ]:
# Population of Barcelona according to the Municipal Register of Inhabitants on January 1 of each year
# https://opendata-ajuntament.barcelona.cat/data/en/dataset/pad_mdbas/resource/eb82adf2-a7b0-40e6-9624-b4b9eff23018
population_df = pd.read_csv("2025_pad_mdbas.csv")
population_df

In [ ]:
population_df["population"] = (
    population_df["Valor"]
        .astype(float)
)

In [ ]:
population_gdf = census_areas_gdf.merge(population_df, on="Seccio_Censal")
population_gdf

In [ ]:
population_gdf.explore("population")

In [ ]:
population_gdf.explore("population", scheme="percentiles")